In [2]:
"""
EEG-to-Text Generation Model with GPT-2 Decoder
Architecture for EEG signal to natural language generation
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Config
from typing import Optional, Tuple


class SimplifiedEEGEncoder(nn.Module):
    """
    Simplified EEG encoder using pure Transformer architecture.
    Replaces GCN+GRU with a cleaner, more effective design.
    """
    def __init__(
        self, 
        num_channels: int = 62,
        time_steps: int = 400,
        d_model: int = 512,
        nhead: int = 8,
        num_layers: int = 4,
        dropout: float = 0.1
    ):
        super().__init__()
        self.num_channels = num_channels
        self.time_steps = time_steps
        self.d_model = d_model
        
        # Spatial embedding: project each channel to d_model
        self.channel_embedding = nn.Linear(time_steps, d_model)
        
        # Positional encoding for channels
        self.channel_pos_encoding = nn.Parameter(
            torch.randn(1, num_channels, d_model) * 0.02
        )
        
        # Transformer encoder to model spatio-temporal dependencies
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-LN for better training stability
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_layers
        )
        
        # Layer norm for output
        self.output_norm = nn.LayerNorm(d_model)
        
        print(f"SimplifiedEEGEncoder: {num_channels} channels -> {d_model}d, {num_layers} layers")
    
    def forward(self, eeg: torch.Tensor) -> torch.Tensor:
        """
        Args:
            eeg: [batch, channels, time_steps]
        Returns:
            encoded: [batch, channels, d_model] - one vector per channel
        """
        batch_size = eeg.shape[0]
        
        # Project temporal dimension to d_model
        # [batch, channels, time] -> [batch, channels, d_model]
        x = self.channel_embedding(eeg)
        
        # Add positional encoding
        x = x + self.channel_pos_encoding
        
        # Apply transformer encoder
        x = self.transformer_encoder(x)
        
        # Output normalization
        x = self.output_norm(x)
        
        return x

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [3]:
class EEGToGPT2Adapter(nn.Module):
    """
    Adapter that projects EEG encoder outputs to GPT-2's input space.
    Uses a learned compression + projection strategy.
    """
    def __init__(
        self,
        eeg_dim: int = 512,
        gpt2_dim: int = 768,
        num_eeg_tokens: int = 8,  # Compress 62 channels to 8 tokens
        dropout: float = 0.1
    ):
        super().__init__()
        self.num_eeg_tokens = num_eeg_tokens
        
        # Learnable query tokens for compression
        self.query_tokens = nn.Parameter(
            torch.randn(1, num_eeg_tokens, eeg_dim) * 0.02
        )
        
        # Cross-attention to compress EEG features
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=eeg_dim,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Project to GPT-2 dimension
        self.projection = nn.Sequential(
            nn.Linear(eeg_dim, gpt2_dim),
            nn.LayerNorm(gpt2_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gpt2_dim, gpt2_dim),
            nn.LayerNorm(gpt2_dim)
        )
        
        print(f"EEGToGPT2Adapter: {eeg_dim}d -> {num_eeg_tokens} tokens -> {gpt2_dim}d")
    
    def forward(self, eeg_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            eeg_features: [batch, num_channels, eeg_dim]
        Returns:
            adapted: [batch, num_eeg_tokens, gpt2_dim]
        """
        batch_size = eeg_features.shape[0]
        
        # Expand query tokens for batch
        queries = self.query_tokens.expand(batch_size, -1, -1)
        
        # Cross-attend to EEG features
        compressed, _ = self.cross_attention(
            query=queries,
            key=eeg_features,
            value=eeg_features
        )
        
        # Project to GPT-2 space
        adapted = self.projection(compressed)
        
        return adapted

In [4]:
class MetadataHead(nn.Module):
    """
    Lightweight metadata prediction head using pooled EEG features.
    """
    def __init__(
        self,
        eeg_dim: int = 512,
        num_colors: int = 12,
        num_objects: int = 90,
        dropout: float = 0.1
    ):
        super().__init__()
        
        # Global pooling
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        # Color classifier
        self.color_head = nn.Sequential(
            nn.Linear(eeg_dim, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(dropout),
            nn.Linear(256, num_colors)
        )
        
        # Object classifier (multi-label)
        self.object_head = nn.Sequential(
            nn.Linear(eeg_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_objects)
        )
    
    def forward(self, eeg_features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            eeg_features: [batch, num_channels, eeg_dim]
        Returns:
            color_logits: [batch, num_colors]
            object_logits: [batch, num_objects]
        """
        # Pool across channels: [batch, eeg_dim, num_channels] -> [batch, eeg_dim, 1]
        pooled = self.pool(eeg_features.transpose(1, 2)).squeeze(-1)
        
        color_logits = self.color_head(pooled)
        object_logits = self.object_head(pooled)
        
        return color_logits, object_logits

In [5]:
class EEGToTextModel(nn.Module):
    """
    Complete EEG-to-Text model with GPT-2 decoder and metadata heads.
    """
    def __init__(
        self,
        num_channels: int = 62,
        time_steps: int = 400,
        num_colors: int = 12,
        num_objects: int = 90,
        gpt2_model_name: str = "gpt2",
        freeze_gpt2_initially: bool = True,
        eeg_encoder_dim: int = 512,
        num_eeg_prefix_tokens: int = 8,
        dropout: float = 0.1
    ):
        super().__init__()
        
        # EEG Encoder
        self.eeg_encoder = SimplifiedEEGEncoder(
            num_channels=num_channels,
            time_steps=time_steps,
            d_model=eeg_encoder_dim,
            dropout=dropout
        )
        
        # Load pretrained GPT-2
        self.gpt2 = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
        gpt2_dim = self.gpt2.config.n_embd
        
        # Freeze GPT-2 initially if requested
        if freeze_gpt2_initially:
            for param in self.gpt2.parameters():
                param.requires_grad = False
            print(f"GPT-2 weights frozen initially")
        
        # Adapter to connect EEG encoder to GPT-2
        self.adapter = EEGToGPT2Adapter(
            eeg_dim=eeg_encoder_dim,
            gpt2_dim=gpt2_dim,
            num_eeg_tokens=num_eeg_prefix_tokens,
            dropout=dropout
        )
        
        # Metadata prediction heads
        self.metadata_head = MetadataHead(
            eeg_dim=eeg_encoder_dim,
            num_colors=num_colors,
            num_objects=num_objects,
            dropout=dropout
        )
        
        self.num_eeg_prefix_tokens = num_eeg_prefix_tokens
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
    
    def unfreeze_gpt2(self):
        """Unfreeze GPT-2 for end-to-end fine-tuning."""
        for param in self.gpt2.parameters():
            param.requires_grad = True
        print("GPT-2 weights unfrozen")
    
    def forward(
        self,
        eeg: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        return_metadata: bool = True
    ) -> dict:
        """
        Args:
            eeg: [batch, channels, time_steps]
            input_ids: [batch, seq_len] - text token IDs
            attention_mask: [batch, seq_len] - attention mask for text
            labels: [batch, seq_len] - labels for language modeling loss
            return_metadata: whether to compute metadata predictions
            
        Returns:
            dict with keys: 'loss', 'text_loss', 'logits', 'color_logits', 'object_logits'
        """
        batch_size = eeg.shape[0]
        
        # Encode EEG
        eeg_features = self.eeg_encoder(eeg)  # [batch, channels, eeg_dim]
        
        # Get metadata predictions (always computed, but loss optional)
        color_logits, object_logits = None, None
        if return_metadata:
            color_logits, object_logits = self.metadata_head(eeg_features)
        
        # Adapt EEG to GPT-2 input space
        eeg_prefix = self.adapter(eeg_features)  # [batch, num_eeg_tokens, gpt2_dim]
        
        # Get text embeddings
        text_embeds = self.gpt2.transformer.wte(input_ids)  # [batch, seq_len, gpt2_dim]
        
        # Concatenate EEG prefix with text embeddings
        inputs_embeds = torch.cat([eeg_prefix, text_embeds], dim=1)
        # [batch, num_eeg_tokens + seq_len, gpt2_dim]
        
        # Create attention mask for full sequence
        if attention_mask is None:
            attention_mask = torch.ones(
                batch_size, input_ids.shape[1], 
                dtype=torch.long, device=input_ids.device
            )
        
        # Extend attention mask for EEG prefix (always attend)
        eeg_attention_mask = torch.ones(
            batch_size, self.num_eeg_prefix_tokens,
            dtype=torch.long, device=attention_mask.device
        )
        full_attention_mask = torch.cat([eeg_attention_mask, attention_mask], dim=1)
        
        # Adjust labels (shift by num_eeg_tokens)
        full_labels = None
        if labels is not None:
            # Pad labels with -100 (ignore) for EEG prefix positions
            eeg_labels = torch.full(
                (batch_size, self.num_eeg_prefix_tokens),
                -100, dtype=torch.long, device=labels.device
            )
            full_labels = torch.cat([eeg_labels, labels], dim=1)
        
        # Forward through GPT-2
        outputs = self.gpt2(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attention_mask,
            labels=full_labels,
            return_dict=True
        )
        
        result = {
            'logits': outputs.logits,
            'text_loss': outputs.loss if labels is not None else None,
            'color_logits': color_logits,
            'object_logits': object_logits
        }
        
        return result
    
    @torch.no_grad()
    def generate(
        self,
        eeg: torch.Tensor,
        tokenizer,
        max_length: int = 50,
        num_beams: int = 4,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 0.95,
        repetition_penalty: float = 1.2
    ) -> list:
        """
        Generate text from EEG using beam search.
        
        Args:
            eeg: [batch, channels, time_steps]
            tokenizer: HuggingFace tokenizer
            max_length: maximum generation length
            num_beams: number of beams for beam search
            
        Returns:
            List of generated text strings
        """
        self.eval()
        batch_size = eeg.shape[0]
        device = eeg.device
        
        # Encode EEG and get prefix
        eeg_features = self.eeg_encoder(eeg)
        eeg_prefix = self.adapter(eeg_features)  # [batch, num_eeg_tokens, gpt2_dim]
        
        # Start with BOS token
        input_ids = torch.full(
            (batch_size, 1),
            tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id,
            dtype=torch.long,
            device=device
        )
        
        # Generate
        generated_ids = self.gpt2.generate(
            input_ids=input_ids,
            inputs_embeds=eeg_prefix,
            max_length=max_length + self.num_eeg_prefix_tokens,
            num_beams=num_beams,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id,
            do_sample=num_beams == 1,
            early_stopping=True
        )
        
        # Remove EEG prefix tokens and decode
        generated_ids = generated_ids[:, self.num_eeg_prefix_tokens:]
        generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        
        return generated_texts

In [6]:
"""
Three-Phase Training Strategy for EEG-to-Text Model
Phase 1: Encoder + Adapter only (GPT-2 frozen)
Phase 2: Add metadata heads
Phase 3: End-to-end fine-tuning (unfreeze GPT-2)
"""

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
import time
import json
from pathlib import Path

# Import your models
# from models import EEGToTextModel


class EEGTextDataset(Dataset):
    """Simple dataset for EEG and text."""
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))
        
        return eeg, meta, text


def collate_fn(batch, pad_id):
    """Collate batch with padding."""
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, text in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(text)
    
    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=pad_id)
    
    return eeg_batch, meta_batch, text_padded


class ThreePhaseTrainer:
    """
    Implements the three-phase training strategy:
    1. Train encoder + adapter (GPT-2 frozen, text-only)
    2. Add metadata heads (GPT-2 still frozen)
    3. End-to-end fine-tuning (unfreeze GPT-2)
    """
    
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        tokenizer,
        device,
        output_dir="./checkpoints",
        num_colors=12,
        num_objects=90
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.tokenizer = tokenizer
        self.device = device
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.num_colors = num_colors
        self.num_objects = num_objects
        
        # Loss functions
        self.text_criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
        self.color_criterion = nn.CrossEntropyLoss()
        self.object_criterion = nn.BCEWithLogitsLoss()
        
        self.history = {
            'phase1': [], 'phase2': [], 'phase3': []
        }
    
    def train_one_epoch(
        self, 
        optimizer, 
        scheduler,
        use_metadata=False,
        text_weight=1.0,
        color_weight=0.1,
        object_weight=0.1,
        phase_name="training"
    ):
        """Train for one epoch."""
        self.model.train()
        total_loss = 0.0
        total_text_loss = 0.0
        total_color_loss = 0.0
        total_object_loss = 0.0
        
        pbar = tqdm(self.train_loader, desc=f"{phase_name}")
        
        for eeg, metadata, text in pbar:
            eeg = eeg.to(self.device)
            metadata = metadata.to(self.device)
            text = text.to(self.device)
            
            # Create labels (shift by 1 for language modeling)
            labels = text[:, 1:].contiguous()
            input_ids = text[:, :-1].contiguous()
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = self.model(
                eeg=eeg,
                input_ids=input_ids,
                labels=labels,
                return_metadata=use_metadata
            )
            
            # Calculate losses
            text_loss = outputs['text_loss']
            loss = text_weight * text_loss
            
            color_loss = torch.tensor(0.0, device=self.device)
            object_loss = torch.tensor(0.0, device=self.device)
            
            if use_metadata:
                # Color loss
                color_targets = metadata[:, 0].long()
                color_loss = self.color_criterion(outputs['color_logits'], color_targets)
                
                # Object loss
                object_targets = metadata[:, 1:].float()
                object_loss = self.object_criterion(outputs['object_logits'], object_targets)
                
                loss = loss + color_weight * color_loss + object_weight * object_loss
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            
            # Track losses
            total_loss += loss.item()
            total_text_loss += text_loss.item()
            total_color_loss += color_loss.item()
            total_object_loss += object_loss.item()
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'txt': f'{text_loss.item():.4f}',
                'clr': f'{color_loss.item():.4f}',
                'obj': f'{object_loss.item():.4f}'
            })
        
        n = len(self.train_loader)
        return {
            'loss': total_loss / n,
            'text_loss': total_text_loss / n,
            'color_loss': total_color_loss / n,
            'object_loss': total_object_loss / n
        }
    
    @torch.no_grad()
    def validate(self, use_metadata=False, text_weight=1.0, color_weight=0.1, object_weight=0.1):
        """Validate the model."""
        self.model.eval()
        total_loss = 0.0
        total_text_loss = 0.0
        total_color_loss = 0.0
        total_object_loss = 0.0
        
        pbar = tqdm(self.val_loader, desc="Validation")
        
        for eeg, metadata, text in pbar:
            eeg = eeg.to(self.device)
            metadata = metadata.to(self.device)
            text = text.to(self.device)
            
            labels = text[:, 1:].contiguous()
            input_ids = text[:, :-1].contiguous()
            
            outputs = self.model(
                eeg=eeg,
                input_ids=input_ids,
                labels=labels,
                return_metadata=use_metadata
            )
            
            text_loss = outputs['text_loss']
            loss = text_weight * text_loss
            
            color_loss = torch.tensor(0.0, device=self.device)
            object_loss = torch.tensor(0.0, device=self.device)
            
            if use_metadata:
                color_targets = metadata[:, 0].long()
                color_loss = self.color_criterion(outputs['color_logits'], color_targets)
                
                object_targets = metadata[:, 1:].float()
                object_loss = self.object_criterion(outputs['object_logits'], object_targets)
                
                loss = loss + color_weight * color_loss + object_weight * object_loss
            
            total_loss += loss.item()
            total_text_loss += text_loss.item()
            total_color_loss += color_loss.item()
            total_object_loss += object_loss.item()
        
        n = len(self.val_loader)
        return {
            'loss': total_loss / n,
            'text_loss': total_text_loss / n,
            'color_loss': total_color_loss / n,
            'object_loss': total_object_loss / n
        }
    
    def phase1_encoder_adapter(self, epochs=10, lr=1e-3, warmup_steps=500):
        """
        Phase 1: Train encoder + adapter only (GPT-2 frozen, text-only).
        This teaches the encoder to extract text-relevant features from EEG.
        """
        print("\n" + "="*80)
        print("PHASE 1: Training Encoder + Adapter (GPT-2 frozen, text-only)")
        print("="*80)
        
        # Only optimize encoder and adapter
        params_to_optimize = list(self.model.eeg_encoder.parameters()) + \
                           list(self.model.adapter.parameters())
        
        optimizer = torch.optim.AdamW(params_to_optimize, lr=lr, weight_decay=0.01)
        
        total_steps = len(self.train_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        
        best_val_loss = float('inf')
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            start_time = time.time()
            
            # Train
            train_metrics = self.train_one_epoch(
                optimizer, 
                scheduler,
                use_metadata=False,
                text_weight=1.0,
                phase_name=f"Phase1 Epoch {epoch}"
            )
            
            # Validate
            val_metrics = self.validate(use_metadata=False, text_weight=1.0)
            
            elapsed = time.time() - start_time
            
            print(f"Train Loss: {train_metrics['loss']:.4f} | Text: {train_metrics['text_loss']:.4f}")
            print(f"Val Loss: {val_metrics['loss']:.4f} | Text: {val_metrics['text_loss']:.4f}")
            print(f"Time: {elapsed:.1f}s")
            
            self.history['phase1'].append({
                'epoch': epoch,
                'train': train_metrics,
                'val': val_metrics
            })
            
            # Save best model
            if val_metrics['loss'] < best_val_loss:
                best_val_loss = val_metrics['loss']
                torch.save(
                    self.model.state_dict(),
                    self.output_dir / 'phase1_best.pt'
                )
                print(f"✓ Saved best Phase 1 model (val_loss={best_val_loss:.4f})")
        
        # Load best model for next phase
        self.model.load_state_dict(torch.load(self.output_dir / 'phase1_best.pt'))
        print("\nPhase 1 complete! Loaded best checkpoint.")
    
    def phase2_add_metadata(self, epochs=10, lr=1e-4, warmup_steps=300):
        """
        Phase 2: Train metadata heads (GPT-2 still frozen).
        Uses lower learning rate and balanced loss weights.
        """
        print("\n" + "="*80)
        print("PHASE 2: Adding Metadata Heads (GPT-2 still frozen)")
        print("="*80)
        
        # Optimize encoder, adapter, and NEW metadata heads
        params_to_optimize = list(self.model.eeg_encoder.parameters()) + \
                           list(self.model.adapter.parameters()) + \
                           list(self.model.metadata_head.parameters())
        
        optimizer = torch.optim.AdamW(params_to_optimize, lr=lr, weight_decay=0.01)
        
        total_steps = len(self.train_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        
        # BALANCED LOSS WEIGHTS: Text is still the primary task
        text_weight = 1.0
        color_weight = 0.2
        object_weight = 0.3
        
        best_val_loss = float('inf')
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            start_time = time.time()
            
            train_metrics = self.train_one_epoch(
                optimizer,
                scheduler,
                use_metadata=True,
                text_weight=text_weight,
                color_weight=color_weight,
                object_weight=object_weight,
                phase_name=f"Phase2 Epoch {epoch}"
            )
            
            val_metrics = self.validate(
                use_metadata=True,
                text_weight=text_weight,
                color_weight=color_weight,
                object_weight=object_weight
            )
            
            elapsed = time.time() - start_time
            
            print(f"Train - Loss: {train_metrics['loss']:.4f} | "
                  f"Text: {train_metrics['text_loss']:.4f} | "
                  f"Color: {train_metrics['color_loss']:.4f} | "
                  f"Object: {train_metrics['object_loss']:.4f}")
            print(f"Val - Loss: {val_metrics['loss']:.4f} | "
                  f"Text: {val_metrics['text_loss']:.4f} | "
                  f"Color: {val_metrics['color_loss']:.4f} | "
                  f"Object: {val_metrics['object_loss']:.4f}")
            print(f"Time: {elapsed:.1f}s")
            
            self.history['phase2'].append({
                'epoch': epoch,
                'train': train_metrics,
                'val': val_metrics
            })
            
            if val_metrics['loss'] < best_val_loss:
                best_val_loss = val_metrics['loss']
                torch.save(
                    self.model.state_dict(),
                    self.output_dir / 'phase2_best.pt'
                )
                print(f"✓ Saved best Phase 2 model (val_loss={best_val_loss:.4f})")
        
        self.model.load_state_dict(torch.load(self.output_dir / 'phase2_best.pt'))
        print("\nPhase 2 complete! Loaded best checkpoint.")
    
    def phase3_end_to_end(self, epochs=15, lr=5e-6, warmup_steps=500):
        """
        Phase 3: End-to-end fine-tuning (unfreeze GPT-2).
        Uses very low learning rate with differential LRs.
        """
        print("\n" + "="*80)
        print("PHASE 3: End-to-End Fine-Tuning (GPT-2 unfrozen)")
        print("="*80)
        
        # Unfreeze GPT-2
        self.model.unfreeze_gpt2()
        
        # Differential learning rates
        param_groups = [
            {'params': self.model.gpt2.parameters(), 'lr': lr},  # Lowest LR
            {'params': self.model.eeg_encoder.parameters(), 'lr': lr * 5},
            {'params': self.model.adapter.parameters(), 'lr': lr * 10},
            {'params': self.model.metadata_head.parameters(), 'lr': lr * 10}
        ]
        
        optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)
        
        total_steps = len(self.train_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        
        # Same balanced weights as Phase 2
        text_weight = 1.0
        color_weight = 0.2
        object_weight = 0.3
        
        best_val_loss = float('inf')
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            start_time = time.time()
            
            train_metrics = self.train_one_epoch(
                optimizer,
                scheduler,
                use_metadata=True,
                text_weight=text_weight,
                color_weight=color_weight,
                object_weight=object_weight,
                phase_name=f"Phase3 Epoch {epoch}"
            )
            
            val_metrics = self.validate(
                use_metadata=True,
                text_weight=text_weight,
                color_weight=color_weight,
                object_weight=object_weight
            )
            
            elapsed = time.time() - start_time
            
            print(f"Train - Loss: {train_metrics['loss']:.4f} | "
                  f"Text: {train_metrics['text_loss']:.4f} | "
                  f"Color: {train_metrics['color_loss']:.4f} | "
                  f"Object: {train_metrics['object_loss']:.4f}")
            print(f"Val - Loss: {val_metrics['loss']:.4f} | "
                  f"Text: {val_metrics['text_loss']:.4f} | "
                  f"Color: {val_metrics['color_loss']:.4f} | "
                  f"Object: {val_metrics['object_loss']:.4f}")
            print(f"Time: {elapsed:.1f}s | LR: {optimizer.param_groups[0]['lr']:.2e}")
            
            self.history['phase3'].append({
                'epoch': epoch,
                'train': train_metrics,
                'val': val_metrics
            })
            
            if val_metrics['loss'] < best_val_loss:
                best_val_loss = val_metrics['loss']
                torch.save(
                    self.model.state_dict(),
                    self.output_dir / 'phase3_best.pt'
                )
                torch.save(
                    self.model.state_dict(),
                    self.output_dir / 'final_model.pt'
                )
                print(f"✓ Saved best Phase 3 model (val_loss={best_val_loss:.4f})")
        
        print("\nPhase 3 complete! Training finished.")
        
        # Save training history
        with open(self.output_dir / 'training_history.json', 'w') as f:
            json.dump(self.history, f, indent=2)

In [9]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
import time
import json
from pathlib import Path

# ... (Keep the EEGTextDataset, collate_fn, and ThreePhaseTrainer classes here) ...

def main():
    """Main training script."""
    
    # Configuration
    H5_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
    MODEL_PATH = "/home/poorna/models/bert-base-uncased"
    CHECKPOINT_DIR = "./checkpoints"
    PHASE_1_CHECKPOINT = Path(CHECKPOINT_DIR) / "phase1_best.pt"
    
    BATCH_SIZE = 16
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {DEVICE}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Create dataset
    dataset = EEGTextDataset(H5_PATH)
    n_train = int(len(dataset) * 0.8)
    n_val = int(len(dataset) * 0.1)
    n_test = len(dataset) - n_train - n_val
    
    train_ds, val_ds, test_ds = random_split(
        dataset, 
        [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
        num_workers=0,  # <-- Fix: Prevents deadlock
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
        num_workers=0,  # <-- Fix: Prevents deadlock
        pin_memory=True
    )
    
    # Create model
    model = EEGToTextModel(
        num_channels=62,
        time_steps=400,  # <-- Corrected time_steps
        num_colors=12,
        num_objects=90,
        gpt2_model_name="gpt2",
        freeze_gpt2_initially=True,
        eeg_encoder_dim=512,
        num_eeg_prefix_tokens=8
    ).to(DEVICE)
    
    
    # --- LOGIC TO SKIP PHASE 1 ---
    skip_phase_1 = False
    if PHASE_1_CHECKPOINT.exists():
        print(f"Found checkpoint: {PHASE_1_CHECKPOINT}")
        print("Loading weights and skipping Phase 1...")
        model.load_state_dict(torch.load(PHASE_1_CHECKPOINT))
        skip_phase_1 = True
    else:
        print("No Phase 1 checkpoint found. Starting from scratch.")
    # -----------------------------

    # Create trainer
    trainer = ThreePhaseTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        tokenizer=tokenizer,
        device=DEVICE,
        output_dir=CHECKPOINT_DIR
    )
    
    # Run three-phase training
    if not skip_phase_1:
        trainer.phase1_encoder_adapter(epochs=10, lr=1e-3)
    else:
        print("Phase 1 already complete.")
        
    print("\nStarting Phase 2...")
    trainer.phase2_add_metadata(epochs=10, lr=1e-4)
    
    print("\nStarting Phase 3...")
    trainer.phase3_end_to_end(epochs=15, lr=5e-6)
    
    print("\n" + "="*80)
    print("TRAINING COMPLETE!")
    print("="*80)
    print(f"Best model saved to: {trainer.output_dir / 'final_model.pt'}")


if __name__ == "__main__":
    main()

Using device: cuda
SimplifiedEEGEncoder: 62 channels -> 512d, 4 layers
GPT-2 weights frozen initially
EEGToGPT2Adapter: 512d -> 8 tokens -> 768d
Total parameters: 139,774,566
Trainable parameters: 15,334,758
Found checkpoint: checkpoints/phase1_best.pt
Loading weights and skipping Phase 1...
Phase 1 already complete.

Starting Phase 2...

PHASE 2: Adding Metadata Heads (GPT-2 still frozen)

Epoch 1/10


Phase2 Epoch 1:   0%|          | 0/1400 [00:00<?, ?it/s]

Validation:   0%|          | 0/175 [00:00<?, ?it/s]

Train - Loss: 2.3414 | Text: 1.8531 | Color: 2.2446 | Object: 0.1313
Val - Loss: 2.2084 | Text: 1.7365 | Color: 2.2177 | Object: 0.0946
Time: 210.3s
✓ Saved best Phase 2 model (val_loss=2.2084)

Epoch 2/10


Phase2 Epoch 2:   0%|          | 0/1400 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/175 [00:00<?, ?it/s]

Train - Loss: 0.9503 | Text: 0.4817 | Color: 2.2006 | Object: 0.0947
Val - Loss: 0.8526 | Text: 0.3822 | Color: 2.2111 | Object: 0.0942
Time: 349.6s | LR: 5.73e-08
✓ Saved best Phase 3 model (val_loss=0.8526)

Epoch 15/15


Phase3 Epoch 15:   0%|          | 0/1400 [00:00<?, ?it/s]

Validation:   0%|          | 0/175 [00:00<?, ?it/s]

Train - Loss: 0.9490 | Text: 0.4806 | Color: 2.1998 | Object: 0.0947
Val - Loss: 0.8524 | Text: 0.3819 | Color: 2.2111 | Object: 0.0942
Time: 364.9s | LR: 0.00e+00
✓ Saved best Phase 3 model (val_loss=0.8524)

Phase 3 complete! Training finished.

TRAINING COMPLETE!
Best model saved to: checkpoints/final_model.pt


In [12]:
"""
Evaluation script for EEG-to-Text model.
Computes BLEU, ROUGE, and METEOR scores.
"""

import torch
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import evaluate as hf_evaluate
from collections import defaultdict
import json

class EEGTextDataset(Dataset):
    """Dataset for evaluation."""
    def __init__(self, h5_path, indices=None):
        self.h5_path = h5_path
        self.h5_file = None
        self.indices = indices
        
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
        
        if self.indices is None:
            self.indices = list(range(self.n_samples))
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        real_idx = self.indices[idx]
        eeg = torch.from_numpy(self.h5_file['eeg'][real_idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][real_idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][real_idx].astype(np.int64))
        
        return eeg, meta, text


class EEGToTextEvaluator:
    """
    Comprehensive evaluator for EEG-to-Text model.
    """
    
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        
        # Load metrics from HuggingFace evaluate
        self.bleu_metric = hf_evaluate.load('bleu')
        self.rouge_metric = hf_evaluate.load('rouge')
        self.meteor_metric = hf_evaluate.load('meteor')
        
        print("Loaded BLEU, ROUGE, and METEOR metrics")
    
    @torch.no_grad()
    def generate_and_evaluate(
        self,
        dataloader,
        max_length=50,
        num_beams=4,
        temperature=1.0,
        top_k=50,
        top_p=0.95
    ):
        """
        Generate text for all samples and compute metrics.
        
        Returns:
            dict with metric scores and generated examples
        """
        self.model.eval()
        
        all_predictions = []
        all_references = []
        all_color_preds = []
        all_color_targets = []
        all_object_preds = []
        all_object_targets = []
        
        print("Generating predictions...")
        for eeg_batch, meta_batch, text_batch in tqdm(dataloader):
            eeg_batch = eeg_batch.to(self.device)
            meta_batch = meta_batch.to(self.device)
            
            # Generate text
            generated_texts = self.model.generate(
                eeg=eeg_batch,
                tokenizer=self.tokenizer,
                max_length=max_length,
                num_beams=num_beams,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p
            )
            
            # Get reference texts
            reference_texts = self.tokenizer.batch_decode(
                text_batch, 
                skip_special_tokens=True
            )
            
            # Get metadata predictions
            eeg_features = self.model.eeg_encoder(eeg_batch)
            color_logits, object_logits = self.model.metadata_head(eeg_features)
            
            color_preds = color_logits.argmax(dim=-1).cpu().numpy()
            object_preds = (torch.sigmoid(object_logits) > 0.5).cpu().numpy()
            
            # Store results
            all_predictions.extend(generated_texts)
            all_references.extend(reference_texts)
            all_color_preds.extend(color_preds)
            all_color_targets.extend(meta_batch[:, 0].cpu().numpy())
            all_object_preds.extend(object_preds)
            all_object_targets.extend(meta_batch[:, 1:].cpu().numpy())
        
        # Compute text generation metrics
        print("\nComputing metrics...")
        text_metrics = self.compute_text_metrics(all_predictions, all_references)
        
        # Compute metadata metrics
        metadata_metrics = self.compute_metadata_metrics(
            all_color_preds,
            all_color_targets,
            all_object_preds,
            all_object_targets
        )
        
        # Combine all metrics
        results = {
            'text_metrics': text_metrics,
            'metadata_metrics': metadata_metrics,
            'num_samples': len(all_predictions)
        }
        
        # Sample some examples
        results['examples'] = []
        for i in range(min(10, len(all_predictions))):
            results['examples'].append({
                'reference': all_references[i],
                'prediction': all_predictions[i],
                'color_target': int(all_color_targets[i]),
                'color_pred': int(all_color_preds[i]),
            })
        
        return results
    
    def compute_text_metrics(self, predictions, references):
        """
        Compute BLEU, ROUGE, and METEOR scores.
        
        Args:
            predictions: List of generated strings
            references: List of reference strings
        """
        # Prepare references in the format expected by metrics
        # Each reference should be a list (for multiple references per prediction)
        references_list = [[ref] for ref in references]
        
        # Compute BLEU (1-4 grams)
        bleu_results = self.bleu_metric.compute(
            predictions=predictions,
            references=references_list,
            max_order=4
        )
        
        # Compute ROUGE
        rouge_results = self.rouge_metric.compute(
            predictions=predictions,
            references=references
        )
        
        # Compute METEOR
        meteor_results = self.meteor_metric.compute(
            predictions=predictions,
            references=references
        )
        
        metrics = {
            'bleu': bleu_results['bleu'],
            'bleu_1': bleu_results['precisions'][0],
            'bleu_2': bleu_results['precisions'][1],
            'bleu_3': bleu_results['precisions'][2],
            'bleu_4': bleu_results['precisions'][3],
            'rouge1': rouge_results['rouge1'],
            'rouge2': rouge_results['rouge2'],
            'rougeL': rouge_results['rougeL'],
            'meteor': meteor_results['meteor']
        }
        
        return metrics
    
    def compute_metadata_metrics(
        self,
        color_preds,
        color_targets,
        object_preds,
        object_targets
    ):
        """Compute accuracy for color and F1 for objects."""
        color_preds = np.array(color_preds)
        color_targets = np.array(color_targets)
        object_preds = np.array(object_preds)
        object_targets = np.array(object_targets)
        
        # Color accuracy
        color_acc = (color_preds == color_targets).mean()
        
        # Object F1 (micro and macro)
        # True positives, false positives, false negatives per sample
        tp = (object_preds * object_targets).sum()
        fp = (object_preds * (1 - object_targets)).sum()
        fn = ((1 - object_preds) * object_targets).sum()
        
        # Micro F1
        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1_micro = 2 * precision * recall / (precision + recall + 1e-10)
        
        # Per-class F1 (macro)
        f1_per_class = []
        for i in range(object_preds.shape[1]):
            tp_i = (object_preds[:, i] * object_targets[:, i]).sum()
            fp_i = (object_preds[:, i] * (1 - object_targets[:, i])).sum()
            fn_i = ((1 - object_preds[:, i]) * object_targets[:, i]).sum()
            
            prec_i = tp_i / (tp_i + fp_i + 1e-10)
            rec_i = tp_i / (tp_i + fn_i + 1e-10)
            f1_i = 2 * prec_i * rec_i / (prec_i + rec_i + 1e-10)
            f1_per_class.append(f1_i)
        
        f1_macro = np.mean(f1_per_class)
        
        return {
            'color_accuracy': float(color_acc),
            'object_f1_micro': float(f1_micro),
            'object_f1_macro': float(f1_macro),
            'object_precision': float(precision),
            'object_recall': float(recall)
        }
    
    def print_results(self, results):
        """Pretty print evaluation results."""
        print("\n" + "="*80)
        print("EVALUATION RESULTS")
        print("="*80)
        
        print(f"\nNumber of samples evaluated: {results['num_samples']}")
        
        print("\n--- Text Generation Metrics ---")
        tm = results['text_metrics']
        print(f"BLEU-1: {tm['bleu_1']:.4f}")
        print(f"BLEU-2: {tm['bleu_2']:.4f}")
        print(f"BLEU-3: {tm['bleu_3']:.4f}")
        print(f"BLEU-4: {tm['bleu_4']:.4f}")
        print(f"BLEU (overall): {tm['bleu']:.4f}")
        print(f"\nROUGE-1: {tm['rouge1']:.4f}")
        print(f"ROUGE-2: {tm['rouge2']:.4f}")
        print(f"ROUGE-L: {tm['rougeL']:.4f}")
        print(f"\nMETEOR: {tm['meteor']:.4f}")
        
        print("\n--- Metadata Prediction Metrics ---")
        mm = results['metadata_metrics']
        print(f"Color Accuracy: {mm['color_accuracy']:.4f}")
        print(f"Object F1 (micro): {mm['object_f1_micro']:.4f}")
        print(f"Object F1 (macro): {mm['object_f1_macro']:.4f}")
        print(f"Object Precision: {mm['object_precision']:.4f}")
        print(f"Object Recall: {mm['object_recall']:.4f}")
        
        print("\n--- Sample Predictions ---")
        for i, example in enumerate(results['examples'][:5]):
            print(f"\nExample {i+1}:")
            print(f"  Reference: {example['reference']}")
            print(f"  Prediction: {example['prediction']}")
            print(f"  Color (target/pred): {example['color_target']} / {example['color_pred']}")
        
        print("\n" + "="*80)


def main():
    """Main evaluation script."""
    
    # Configuration
    H5_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
    MODEL_PATH = "/home/poorna/models/bert-base-uncased"
    CHECKPOINT_PATH = "./checkpoints/final_model.pt"
    BATCH_SIZE = 16
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {DEVICE}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Create test dataset (last 10%)
    full_dataset = EEGTextDataset(H5_PATH)
    n_total = len(full_dataset)
    n_train = int(n_total * 0.8)
    n_val = int(n_total * 0.1)
    test_indices = list(range(n_train + n_val, n_total))
    
    test_dataset = EEGTextDataset(H5_PATH, indices=test_indices)
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4
    )
    
    print(f"Test set size: {len(test_dataset)} samples")
    
    # Load model
    print(f"\nLoading model from {CHECKPOINT_PATH}")
    model = EEGToTextModel(
        num_channels=62,
        time_steps=400,
        num_colors=12,
        num_objects=90,
        gpt2_model_name="gpt2",
        freeze_gpt2_initially=False,
        eeg_encoder_dim=512,
        num_eeg_prefix_tokens=8
    ).to(DEVICE)
    
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    print("Model loaded successfully")
    
    # Create evaluator
    evaluator = EEGToTextEvaluator(model, tokenizer, DEVICE)
    
    # Run evaluation with different generation strategies
    print("\n" + "="*80)
    print("BEAM SEARCH (num_beams=4)")
    print("="*80)
    
    results_beam = evaluator.generate_and_evaluate(
        test_loader,
        max_length=50,
        num_beams=4,
        temperature=1.0
    )
    
    evaluator.print_results(results_beam)
    
    # Save results
    output_path = "./checkpoints/evaluation_results.json"
    with open(output_path, 'w') as f:
        # Convert numpy types to Python types for JSON serialization
        results_to_save = {
            'beam_search': results_beam
        }
        json.dump(results_to_save, f, indent=2, default=float)
    
    print(f"\nResults saved to {output_path}")
    
    # Optional: Try sampling for diversity
    print("\n" + "="*80)
    print("SAMPLING (top_k=50, top_p=0.95)")
    print("="*80)
    
    results_sample = evaluator.generate_and_evaluate(
        test_loader,
        max_length=50,
        num_beams=1,  # No beam search
        temperature=0.8,
        top_k=50,
        top_p=0.95
    )
    
    evaluator.print_results(results_sample)


if __name__ == "__main__":
    main()

Using device: cuda
Test set size: 2800 samples

Loading model from ./checkpoints/final_model.pt
SimplifiedEEGEncoder: 62 channels -> 512d, 4 layers
EEGToGPT2Adapter: 512d -> 8 tokens -> 768d
Total parameters: 139,774,566
Trainable parameters: 139,774,566
Model loaded successfully


[nltk_data] Downloading package wordnet to /home/poorna/nltk_data...
[nltk_data] Downloading package punkt_tab to /home/poorna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/poorna/nltk_data...


Loaded BLEU, ROUGE, and METEOR metrics

BEAM SEARCH (num_beams=4)
Generating predictions...


  0%|          | 0/175 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=


Computing metrics...

EVALUATION RESULTS

Number of samples evaluated: 2800

--- Text Generation Metrics ---
BLEU-1: 0.9957
BLEU-2: 0.0000
BLEU-3: 0.0000
BLEU-4: 0.0000
BLEU (overall): 0.0000

ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000

METEOR: 0.0435

--- Metadata Prediction Metrics ---
Color Accuracy: 0.2400
Object F1 (micro): 0.0000
Object F1 (macro): 0.0000
Object Precision: 0.0000
Object Recall: 0.0000

--- Sample Predictions ---

Example 1:
  Reference: a bustling city street with tall buildings and moving vehicles.
  Prediction: .
  Color (target/pred): 4 / 4

Example 2:
  Reference: a city street at dusk, illuminated by streetlights and vehicle headlights.
  Prediction: .
  Color (target/pred): 4 / 4

Example 3:
  Reference: aerial view of a modern cityscape with diverse buildings and green spaces.
  Prediction: .
  Color (target/pred): 4 / 4

Example 4:
  Reference: a bustling cityscape with tall buildings and a river, under a partly cloudy sky.
  Prediction: .
  Color (

  0%|          | 0/175 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not va


Computing metrics...

EVALUATION RESULTS

Number of samples evaluated: 2800

--- Text Generation Metrics ---
BLEU-1: 0.1873
BLEU-2: 0.0006
BLEU-3: 0.0000
BLEU-4: 0.0000
BLEU (overall): 0.0000

ROUGE-1: 0.0470
ROUGE-2: 0.0003
ROUGE-L: 0.0445

METEOR: 0.0500

--- Metadata Prediction Metrics ---
Color Accuracy: 0.2400
Object F1 (micro): 0.0000
Object F1 (macro): 0.0000
Object Precision: 0.0000
Object Recall: 0.0000

--- Sample Predictions ---

Example 1:
  Reference: a bustling city street with tall buildings and moving vehicles.
  Prediction: melted, directly the. tablet
  Color (target/pred): 4 / 4

Example 2:
  Reference: a city street at dusk, illuminated by streetlights and vehicle headlights.
  Prediction: dense and. with red, environment
  Color (target/pred): 4 / 4

Example 3:
  Reference: aerial view of a modern cityscape with diverse buildings and green spaces.
  Prediction: near water in lush environment. iced
  Color (target/pred): 4 / 4

Example 4:
  Reference: a bustling ci


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/poorna/venvs/torch/lib64/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/poorna/venvs/torch/lib64/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/poorna/venvs/torch/lib64/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import